# Week 11: Agentic AI — LangGraph
**Applied Generative AI**
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)
*Developed by Prachi Patel, PhD | Open Curriculum — CC BY 4.0*

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Build stateful multi-step agents with LangGraph** — Design graphs with nodes, edges, and state that flows between them
2. **Implement conditional routing and branching** — Route queries to different specialist nodes based on query type
3. **Design human-in-the-loop agent workflows** — Add approval nodes, checkpoint/resume patterns, and understand when human oversight is critical
4. **Evaluate agent behavior systematically** — Track trajectories, measure routing accuracy, and compare performance across query types

---
## ⚙️ Setup — Install & Import

In [ ]:
!pip install -q langgraph langchain langchain-community sentence-transformers faiss-cpu transformers
# For graph visualization in Colab:
!apt-get -qq install -y graphviz libgraphviz-dev 2>/dev/null || true
!pip install -q pygraphviz 2>/dev/null || true

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer
from langgraph.graph import StateGraph, END
from transformers import pipeline
from typing import TypedDict
import random

---
## 1. LangGraph Fundamentals

LangGraph extends LangChain with **stateful, cyclic graphs**. Unlike simple chains, graphs support:
- **State** that flows between nodes
- **Conditional edges** for branching
- **Cycles** for iterative refinement

### StateGraph, Nodes, and Edges

A **StateGraph** defines:
1. **State schema** — A TypedDict describing what data flows between nodes
2. **Nodes** — Functions that receive state, process it, and return updated state
3. **Edges** — Connections between nodes (fixed or conditional)

In [ ]:
# TypedDict for state management — defines the "shape" of data flowing through the graph
class SimpleState(TypedDict):
    query: str
    answer: str
    selected_node: str

# Nodes are pure functions: state in, state out
def node_a(state: SimpleState):
    return {"answer": f"Node A processed: {state['query']}"}

def node_b(state: SimpleState):
    return {"answer": f"Node B processed: {state['query']}"}

# Router decides which node to go to
def router(state: SimpleState):
    if "a" in state["query"].lower():
        return {"selected_node": "node_a"}
    return {"selected_node": "node_b"}

In [ ]:
# Build a simple graph with conditional edges
workflow = StateGraph(SimpleState)
workflow.add_node("router", router)
workflow.add_node("node_a", node_a)
workflow.add_node("node_b", node_b)

workflow.set_entry_point("router")
workflow.add_conditional_edges("router", lambda s: s["selected_node"], {"node_a": "node_a", "node_b": "node_b"})
workflow.add_edge("node_a", END)
workflow.add_edge("node_b", END)

app = workflow.compile()

# Test
result = app.invoke({"query": "hello world", "answer": "", "selected_node": ""})
print(f"Query: 'hello world' -> {result['answer']}")
result2 = app.invoke({"query": "alpha test", "answer": "", "selected_node": ""})
print(f"Query: 'alpha test' -> {result2['answer']}")

In [ ]:
# Graph visualization
from IPython.display import Image

try:
    graph_image = app.get_graph().draw_png()
    display(Image(graph_image))
except Exception as e:
    print("Graphviz not available. In Colab, run: !apt-get install graphviz && !pip install pygraphviz")
    print("Graph structure:", app.get_graph())

---
## 2. Agentic RAG with LangGraph

We build an **agentic RAG** system: a router decides whether to use retrieval (RAG), answer directly (LLM only), or say "I don't know."

### Harry Potter Knowledge Base

We use a small knowledge base about the Harry Potter universe. The router sends fact-based queries to RAG and general queries to the simple node.

In [ ]:
# Harry Potter knowledge base (from Lecture3_agentic_rag)
knowledge_base = [
    "Expelliarmus is a disarming spell that forces your opponent to release whatever they are holding.",
    "Lumos is a spell that creates light from the tip of the caster's wand.",
    "Wingardium Leviosa is a levitation spell used to make objects fly.",
    "Harry Potter's best friends are Hermione Granger and Ron Weasley.",
    "Hogwarts has four houses: Gryffindor, Hufflepuff, Ravenclaw, and Slytherin.",
    "Albus Dumbledore was the headmaster of Hogwarts.",
    "The main antagonist is Lord Voldemort.",
    "Quidditch is a sport played on flying broomsticks.",
    "A Patronus is a charm that summons a magical guardian, often in the form of an animal."
]

# Embeddings + FAISS
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(knowledge_base)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

def retrieve(query, top_k=2):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, top_k)
    return [knowledge_base[i] for i in indices[0]]

# LLM (open-source, no API key)
generator = pipeline("text-generation", model="EleutherAI/gpt-neo-1.3B")

def call_llm(prompt, max_new_tokens=50):
    output = generator(prompt, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
    return output[0]['generated_text'][len(prompt):].strip()

In [ ]:
# Router: RAG vs direct answer vs "I don't know"
class HPState(TypedDict):
    query: str
    answer: str
    selected_node: str

def router_node(state: HPState):
    q = state['query'].lower()
    # RAG: spell names, characters, Hogwarts facts
    if any(w in q for w in ["spell", "lumos", "expelliarmus", "leviosa", "patronus", "quidditch", "dumbledore", "voldemort", "hogwarts", "hermione", "ron"]):
        return {"selected_node": "rag_node"}
    # Out-of-scope: topics not in our KB
    if any(w in q for w in ["real world", "physics", "math", "unrelated"]):
        return {"selected_node": "dont_know_node"}
    # General: jokes, greetings, etc.
    return {"selected_node": "simple_node"}

# RAG node: retrieve + generate
def rag_node(state: HPState):
    print("  → RAG node activated")
    context = "\n".join(retrieve(state['query']))
    prompt = f"You are a Hogwarts professor. Use this context to answer.\n\nContext:\n{context}\n\nQuestion: {state['query']}\nAnswer:"
    return {"answer": call_llm(prompt)}

# Simple node: LLM only
def simple_node(state: HPState):
    print("  → Simple node activated")
    prompt = f"Answer like a Hogwarts professor.\n{state['query']}\nAnswer:"
    return {"answer": call_llm(prompt)}

# Don't know node
def dont_know_node(state: HPState):
    print("  → Don't know node activated")
    return {"answer": "I'm sorry, that's outside my knowledge of the wizarding world. I can only help with Hogwarts and magical topics."}

In [ ]:
# Build the complete graph with conditional routing
hp_workflow = StateGraph(HPState)
hp_workflow.add_node("router", router_node)
hp_workflow.add_node("rag_node", rag_node)
hp_workflow.add_node("simple_node", simple_node)
hp_workflow.add_node("dont_know_node", dont_know_node)

hp_workflow.set_entry_point("router")
hp_workflow.add_conditional_edges("router", lambda s: s['selected_node'], {
    "rag_node": "rag_node", "simple_node": "simple_node", "dont_know_node": "dont_know_node"
})
hp_workflow.add_edge("rag_node", END)
hp_workflow.add_edge("simple_node", END)
hp_workflow.add_edge("dont_know_node", END)

hp_app = hp_workflow.compile()

In [ ]:
# Test with various queries
test_queries = [
    "What is Lumos?",
    "Who is the headmaster of Hogwarts?",
    "Tell me a short joke about wizards.",
    "What is the capital of France?"  # Out of scope
]

for q in test_queries:
    result = hp_app.invoke({"query": q, "answer": "", "selected_node": ""})
    print(f"Q: {q}")
    print(f"A: {result['answer']}\n{'-'*50}")

---
## 3. Conditional Routing and Branching

Multi-path routing sends queries to **specialist nodes** based on type. We incorporate the **Space Travel Assistant** concept with RAG, fun facts, and travel tips.

In [ ]:
# Space Travel Assistant — multi-path routing
space_knowledge = [
    "Mars: the red planet, known for its iron oxide dust and potential for future human colonization.",
    "Jupiter: the largest planet in the Solar System with a massive storm called the Great Red Spot.",
    "Saturn: famous for its stunning ring system made mostly of ice particles and rock debris.",
    "Voyager 1: a spacecraft launched in 1977 that is now in interstellar space.",
    "Apollo 11: the first successful mission to land humans on the Moon in 1969."
]

# Build FAISS index for space KB
space_embeddings = model.encode(space_knowledge)
space_index = faiss.IndexFlatL2(space_embeddings.shape[1])
space_index.add(space_embeddings)

def retrieve_space(query, top_k=2):
    qe = model.encode([query])
    _, indices = space_index.search(qe, top_k)
    return [space_knowledge[i] for i in indices[0]]

In [ ]:
class SpaceState(TypedDict):
    query: str
    answer: str
    selected_node: str

def space_router(state: SpaceState):
    q = state['query'].lower()
    if any(w in q for w in ["mars", "jupiter", "saturn", "voyager", "apollo", "moon", "planet", "rings"]):
        return {"selected_node": "rag_node"}
    if "fun" in q or "fact" in q or "interesting" in q:
        return {"selected_node": "fun_node"}
    if "travel" in q or "how can i" in q or "get to" in q:
        return {"selected_node": "travel_tips_node"}
    return {"selected_node": "simple_node"}

def space_rag_node(state: SpaceState):
    print("  → RAG node (space facts)")
    context = "\n".join(retrieve_space(state['query']))
    prompt = f"You are a space expert. Use this context.\n\nContext:\n{context}\n\nQuestion: {state['query']}\nAnswer:"
    return {"answer": call_llm(prompt)}

def fun_node(state: SpaceState):
    print("  → Fun node")
    prompt = f"Give one fun fact about space or astronomy. Be brief.\n"
    return {"answer": call_llm(prompt)}

def travel_tips_node(state: SpaceState):
    print("  → Travel tips node")
    context = "\n".join(retrieve_space(state['query']))
    prompt = f"Give imaginative travel advice for space. Context:\n{context}\n\nQuestion: {state['query']}\nAnswer:"
    return {"answer": call_llm(prompt)}

def space_simple_node(state: SpaceState):
    print("  → Simple node")
    prompt = f"Answer as a space enthusiast.\n{state['query']}\nAnswer:"
    return {"answer": call_llm(prompt)}

In [ ]:
# Build Space Travel graph with 4-way conditional routing
space_workflow = StateGraph(SpaceState)
space_workflow.add_node("router", space_router)
space_workflow.add_node("rag_node", space_rag_node)
space_workflow.add_node("fun_node", fun_node)
space_workflow.add_node("travel_tips_node", travel_tips_node)
space_workflow.add_node("simple_node", space_simple_node)

space_workflow.set_entry_point("router")
space_workflow.add_conditional_edges("router", lambda s: s['selected_node'], {
    "rag_node": "rag_node", "fun_node": "fun_node",
    "travel_tips_node": "travel_tips_node", "simple_node": "simple_node"
})
for n in ["rag_node", "fun_node", "travel_tips_node", "simple_node"]:
    space_workflow.add_edge(n, END)

space_app = space_workflow.compile()

# Test
for q in ["Tell me about Saturn's rings.", "Give me a fun fact about space!", "How can I travel to Mars?"]:
    r = space_app.invoke({"query": q, "answer": "", "selected_node": ""})
    print(f"Q: {q}\nA: {r['answer']}\n{'-'*50}")

---
## 4. Human-in-the-Loop

For sensitive domains, we add an **approval node** that pauses for human confirmation. LangGraph supports checkpointing for resume.

In [ ]:
# Human-in-the-loop: approval node
class ApprovalState(TypedDict):
    query: str
    answer: str
    approved: bool
    needs_approval: bool

def generate_node(state: ApprovalState):
    # Simulate generating a sensitive response
    state['answer'] = f"[Generated response for: {state['query']}]"
    state['needs_approval'] = True
    return state

def approval_node(state: ApprovalState):
    """Pause for human confirmation. In Colab/notebook, we simulate with a flag."""
    print(f"  → Pending approval for: {state['query'][:50]}...")
    # In production: await human input. Here we auto-approve for demo.
    state['approved'] = True
    return state

def finalize_node(state: ApprovalState):
    if state['approved']:
        state['answer'] = state['answer'] + " [APPROVED]"
    else:
        state['answer'] = "Response rejected by human reviewer."
    return state

In [ ]:
# Build HITL graph: generate -> approval -> finalize
hitl_workflow = StateGraph(ApprovalState)
hitl_workflow.add_node("generate", generate_node)
hitl_workflow.add_node("approval", approval_node)
hitl_workflow.add_node("finalize", finalize_node)

hitl_workflow.set_entry_point("generate")
hitl_workflow.add_edge("generate", "approval")
hitl_workflow.add_edge("approval", "finalize")
hitl_workflow.add_edge("finalize", END)

hitl_app = hitl_workflow.compile()

r = hitl_app.invoke({"query": "Sensitive medical query", "answer": "", "approved": False, "needs_approval": False})
print(f"Result: {r['answer']}")
print("\nWhen human oversight is critical: medical, financial, legal, content moderation.")
print("When optional: general Q&A, creative tasks, low-stakes recommendations.")

---
## 5. Evaluating Agent Behavior

We track **trajectories** (which nodes were visited) and measure **routing accuracy** and **answer quality**.

In [ ]:
# Track agent trajectories using stream mode
trajectories = []

for q in ["What is Lumos?", "Tell me a joke.", "What is 2+2?"]:
    events = list(hp_app.stream({"query": q, "answer": "", "selected_node": ""}))
    nodes_visited = []
    for chunk in events:
        if isinstance(chunk, dict):
            nodes_visited.extend(chunk.keys())
        elif isinstance(chunk, tuple):
            nodes_visited.append(chunk[0])
    trajectories.append({"query": q, "nodes": nodes_visited})

for t in trajectories:
    print(f"Query: {t['query']} -> Nodes: {t['nodes']}")

In [ ]:
# Simple evaluation: routing accuracy
eval_data = [
    ("What is Expelliarmus?", "rag_node"),
    ("Tell me a wizard joke", "simple_node"),
    ("Who is Dumbledore?", "rag_node"),
    ("What is quantum physics?", "dont_know_node"),
]

correct = 0
for query, expected_node in eval_data:
    result = hp_app.invoke({"query": query, "answer": "", "selected_node": ""})
    actual = result.get("selected_node", "")
    # If selected_node not in final state, infer from router logic
    if not actual:
        q = query.lower()
        if any(w in q for w in ["spell", "lumos", "expelliarmus", "dumbledore"]):
            actual = "rag_node"
        elif any(w in q for w in ["quantum", "physics"]):
            actual = "dont_know_node"
        else:
            actual = "simple_node"
    match = actual == expected_node
    correct += match
    print(f"{query[:40]:40} -> expected {expected_node:15} got {actual:15} {'✓' if match else '✗'}")

print(f"\nRouting accuracy: {correct}/{len(eval_data)} = {100*correct/len(eval_data):.0f}%")

In [ ]:
# Compare agent performance across query types
def run_and_get_node(query):
    """Run router logic to get selected node."""
    q = query.lower()
    if any(w in q for w in ["spell", "lumos", "expelliarmus", "leviosa", "patronus", "quidditch", "dumbledore", "voldemort", "hogwarts"]):
        return "rag_node"
    if any(w in q for w in ["real world", "physics", "math", "quantum", "unrelated"]):
        return "dont_know_node"
    return "simple_node"

query_types = {"RAG": ["What is Lumos?", "Who is Dumbledore?"], "Simple": ["Tell me a joke", "Hello!"], "Out-of-scope": ["What is 2+2?", "Explain quantum physics"]}

for qtype, queries in query_types.items():
    print(f"\n{qtype} queries:")
    for q in queries:
        node = run_and_get_node(q)
        r = hp_app.invoke({"query": q, "answer": "", "selected_node": ""})
        ans_preview = (r['answer'] or "")[:60] + "..." if len(r.get('answer','')) > 60 else (r.get('answer','') or "")
        print(f"  {q} -> {node} -> {ans_preview}")

---
## 6. Exercises

### Exercise 1: Add a Clarification Node

Add a new specialist node to the agentic RAG system: a **clarification** node that triggers when the query is ambiguous (e.g., "Tell me more" or "What about that?"). The node should ask a follow-up question or request clarification.

In [ ]:
# Your code here
# 1. Add "clarification" to router logic for ambiguous queries
# 2. Implement clarification_node that returns a prompt like "Could you clarify what you'd like to know?"
# 3. Add the node and edge to hp_workflow, recompile, and test

### Exercise 2: Human-in-the-Loop for Sensitive Domain

Build a human-in-the-loop workflow for a sensitive domain (e.g., medical or financial queries). The workflow should:
- Route medical/financial queries to a "generate" node
- Pause at an "approval" node (simulate with a variable)
- Only finalize if approved; otherwise return a safe fallback message

In [ ]:
# Your code here
# Define MedicalState or FinancialState, router, generate, approval, finalize nodes
# Use conditional edge: if approved -> finalize, else -> fallback_safe_message

### Exercise 3: Evaluation Dataset

Create an evaluation dataset of **10 queries** with expected routing (rag_node, simple_node, or dont_know_node). Run your agent on each, record the actual node visited, and compute **routing accuracy**. Report which query types had the most errors.

In [ ]:
# Your code here
# eval_dataset = [(query, expected_node), ...]
# For each: invoke agent, get selected_node from stream/state, compare to expected
# Print accuracy and error analysis

---
## 7. Responsible AI Reflection

> **LangGraph makes it easy to build complex agent workflows, but complexity can hide failures.** A 5-node agent graph with conditional routing has dozens of possible execution paths. How do you test all of them? Is it responsible to deploy an agent whose full behavior space you haven't explored?

*Reflect on: path coverage, edge cases, adversarial queries, and the tension between flexibility and controllability.*

---
## 8. Summary & Next Steps

### Summary

- **LangGraph** — StateGraph, TypedDict state, nodes, conditional edges, visualization
- **Agentic RAG** — Router decides RAG vs direct vs "I don't know"; Harry Potter and Space Travel examples
- **Multi-path routing** — Specialist nodes (RAG, fun, travel tips) based on query type
- **Human-in-the-loop** — Approval nodes, checkpoint/resume, when HITL is critical vs optional
- **Evaluation** — Trajectories, routing accuracy, performance by query type

### Next Steps

- Multi-agent collaboration (agents that delegate to each other)
- Production deployment: persistence, observability, guardrails
- Advanced planning: Plan-and-Execute, ReWOO